In [ ]:
from pathlib import Path
import pandas as pd
from utils_data_analysis import plot_plate_view
from tqdm import tqdm

### Plotting percentage of Mtb infected cells per well based on different subcellular compartments

In [ ]:
# ------ Plotting %Mtb_infected_cells per well
# Loop thorugh .csv in the results folder
results_path = Path(r"./results/siMtb screen I_LØ/sum_infection_results")

csv_list = []

for file_path in results_path.glob(f"*.csv"):
    if "infection" in str(file_path):
        csv_list.append(str(file_path))

columns_to_plot = ["%_inf_cells", "%_inf_membranes", "%_inf_cytoplasms"]

for csv_path in csv_list:

    plate_nr = csv_path.split(".")[0].split("_")[-1]

    df = pd.read_csv(csv_path, index_col=0)

    for column in columns_to_plot:

        plot_plate_view(
            df,
            results_path=results_path,
            column_name=column,
            plate=plate_nr,
            title=f"Percentage of Mtb infected cells per well (based on: {column.split('_')[-1]})",
            label="%_Mtb_inf_cells",
            display=False,
            cmap="magma"
        )


### Plotting total number of cells per well

In [ ]:
# ------ Plotting cell numbers per well
# Loop thorugh .csv in the results folder
results_path = Path(r"./results/siMtb screen I_LØ/sum_infection_results")

csv_list = []

for file_path in results_path.glob(f"*.csv"):
    if "infection" in str(file_path):
        csv_list.append(str(file_path))

csv_list

for csv_path in csv_list:

    plate_nr = csv_path.split(".")[0].split("_")[-1]

    df = pd.read_csv(csv_path, index_col=0)

    plot_plate_view(df, results_path=results_path, column_name="total_nr_cells", plate=plate_nr, title="Cell number per well", label="cell_number", display=False, fmt=0, cmap="magma")

### Plotting Chmp4B, GAL3 and LC3B average punctae in infected and non-infected cells

In [ ]:
# ------ Plotting Mean Chmp4B, GAL3 and LC3B punctae per cell (infected, non-infected, all)

markers = ["Chmp4B", "GAL3", "LC3B"]
infection_groups = [("Infected", "infected"), ("Non-infected", "uninfected"), ("All_cells", "all")]
subcellular_compartments = [("Cell", "Mtb_infected_cell"), ("Membrane", "Mtb_infected_membrane"), ("Cytoplasm", "Mtb_infected_cytoplasm")]

# Loop thorugh .csv in the results folder
results_path = Path(r"./results/siMtb screen I_LØ/per_cell_label_results")

csv_list = []

for file_path in results_path.glob(f"*.csv"):
    if "per_label" in str(file_path):
        csv_list.append(str(file_path))

# Read each .csv and plot the different combinations
for csv_path in tqdm(csv_list):

    # Read once per loop to avoid repeated IO operations
    plate_nr = csv_path.split(".")[0].split("_")[-1]
    df = pd.read_csv(csv_path, index_col=0)

    for population_descriptor, population_id in infection_groups:

        for compartment_descriptor, compartment_filter in subcellular_compartments:

            #Apply infected vs non-infected filtering
            if population_id == "infected":
                df_filtered = df[df[compartment_filter] == True]

            elif population_id == "uninfected":
                df_filtered = df[df[compartment_filter] == False]

            elif population_id == "all": # Regardless of the cellular compartment the results of the plot will be the same (plotting all cells)
                df_filtered = df

            for marker in markers:

                result = (
                    df_filtered.groupby("well_id")
                    .agg(
                        marker_sum=(f"{marker}_num_points", "sum"),
                        ObjectNumber_count=("ObjectNumber", "count")
                    )
                    .reset_index()
                )

                # Add ratio column
                result[f"{marker}_avg_points"] = result["marker_sum"] / result["ObjectNumber_count"]

                plot_plate_view(df=result, 
                                results_path=results_path, 
                                column_name=f"{marker}_avg_points", 
                                plate=plate_nr, 
                                title=f"Mean {marker} punctae per cell - Population: {population_descriptor} - Inf_status_based_on: {compartment_descriptor}" , 
                                label=f"{marker}_avg_punctae",
                                population=population_id,
                                compartment=compartment_descriptor, 
                                display=False, cmap="magma")

### Plotting Mtb bacterial features (area, axis lengths)

In [ ]:
features = ["area", "axis_major_length", "axis_minor_length"]

# Loop thorugh .csv in the results folder
results_path = Path(r"./results/siMtb screen I_LØ/per_bacteria_label_results")

csv_list = []

for file_path in results_path.glob(f"*.csv"):
    if "mtb_results" in str(file_path):
        csv_list.append(str(file_path))

# Read each .csv and plot the different combinations
for csv_path in tqdm(csv_list):

    df = pd.read_csv(csv_path, index_col=0)

    plate_nr = csv_path.split(".")[0].split("_")[-1]

    for feature in features:

        result = (df.groupby("well_id")
                .agg(
                    sum=(feature, "sum"),
                    ObjectNumber_count=("ObjectNumber", "count")
                )
                .reset_index()
        )

        # Add ratio column
        result[f"avg_{feature}"] = result["sum"] / result["ObjectNumber_count"]

        plot_plate_view(df=result, 
                        results_path=results_path, 
                        column_name=f"avg_{feature}", 
                        plate=plate_nr, 
                        title=f"Mean Mtb {feature}" , 
                        label=f"Mtb_{feature}",
                        display=False, cmap="magma")

### Plotting % of Mtb bacteria per well by compartment (per-bacterium location flags)

In [ ]:
# ------ % of Mtb bacteria (ObjectNumber) per well: counts with each location flag / all bacteria in well
results_path = Path(r"./results/siMtb screen I_LØ/per_bacteria_label_results")

csv_list = []
for file_path in results_path.glob("*.csv"):
    if "mtb_results" in str(file_path):
        csv_list.append(str(file_path))


def _as_bool(series):
    if series.dtype == bool:
        return series
    return series.map(lambda x: str(x).lower() in ("true", "1"))


columns_to_plot = [
    "pct_mtb_cytoplasm",
    "pct_mtb_membrane",
    "pct_mtb_extracellular",
    "pct_mtb_membrane_not_cytoplasm",
]

titles = {
    "pct_mtb_cytoplasm": "% Mtb in cytoplasm (location_cytoplasm True)",
    "pct_mtb_membrane": "% Mtb at membrane (location_membrane True)",
    "pct_mtb_extracellular": "% Mtb extracellular (location_extracellular True)",
    "pct_mtb_membrane_not_cytoplasm": "% Mtb membrane & not cytoplasm (membrane True, cytoplasm False)",
}

for csv_path in tqdm(csv_list):
    df = pd.read_csv(csv_path, index_col=0)
    plate_nr = csv_path.split(".")[0].split("_")[-1]

    cy = _as_bool(df["location_cytoplasm"])
    me = _as_bool(df["location_membrane"])
    ex = _as_bool(df["location_extracellular"])
    mem_not_cyto = me & ~cy

    result = (
        df.assign(
            location_cytoplasm=cy,
            location_membrane=me,
            location_extracellular=ex,
            _mem_not_cyto=mem_not_cyto,
        )
        .groupby("well_id", as_index=False)
        .agg(
            n_cytoplasm=("location_cytoplasm", "sum"),
            n_membrane=("location_membrane", "sum"),
            n_extracellular=("location_extracellular", "sum"),
            n_membrane_not_cytoplasm=("_mem_not_cyto", "sum"),
            total=("ObjectNumber", "count"),
        )
    )
    total_safe = result["total"].replace(0, float("nan"))
    n_keys = {
        "pct_mtb_cytoplasm": "n_cytoplasm",
        "pct_mtb_membrane": "n_membrane",
        "pct_mtb_extracellular": "n_extracellular",
        "pct_mtb_membrane_not_cytoplasm": "n_membrane_not_cytoplasm",
    }
    for col in columns_to_plot:
        result[col] = 100 * result[n_keys[col]] / total_safe

    result = result[["well_id"] + columns_to_plot]

    for column in columns_to_plot:
        plot_plate_view(
            df=result,
            results_path=results_path,
            column_name=column,
            plate=plate_nr,
            title=titles[column],
            label="% Mtb bacteria",
            display=False,
            cmap="magma",
        )